# XGLUE - Graph and Spectral Analysis
Validation report for graph extraction, spectral feature coverage, and PSS/Wasserstein score distributions.


In [ ]:
# SPECTRA notebook progress: begin
from tqdm.auto import tqdm

def _spectra_enable_cell_progress():
    from IPython import get_ipython
    _shell = get_ipython()
    if _shell is None:
        return
    _old = getattr(_shell, "_spectra_cell_progress", None)
    if _old is not None:
        _old[0].close()
        for _event, _callback in _old[1:]:
            _shell.events.unregister(_event, _callback)
    _bar = tqdm(total=5, desc="Notebook cells", unit="cell", mininterval=2)
    def _after_cell(_result):
        if _bar.n < _bar.total:
            _bar.update(1)
        if getattr(_result, "error_in_exec", None) or getattr(_result, "error_before_exec", None):
            _bar.set_description("Notebook cell failed")
            _bar.close()
        elif _bar.n >= _bar.total:
            _bar.close()
    _shell.events.register("post_run_cell", _after_cell)
    _shell._spectra_cell_progress = (_bar, ("post_run_cell", _after_cell))

_spectra_enable_cell_progress()
# SPECTRA notebook progress: end

from pathlib import Path
import sys


def _find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "spectral_code").exists() and (candidate / "pipelines").exists():
            return candidate
    raise RuntimeError("Project root not found.")


PROJECT_ROOT = _find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from IPython.display import display

from spectral_code.evaluation.notebook_helpers import (
    artifact_status_dataframe,
    configure_notebook_style,
    dataset_overview_rows,
    display_clone_nonclone_pair_inspection,
    display_code_graph_side_by_side_examples,
    display_dataset_overview,
    display_global_threshold_tuning_summary,
    display_pipeline_validation,
    display_similarity_distribution_report,
    display_statistical_distribution_plots,
    display_tuning_report,
    graph_manifest_summary_dataframe,
    load_pairs_for_spec,
    load_tuning_results,
    notebook_output_dir,
    pair_stats_dataframe,
    plot_graph_coverage_from_timing,
    plot_graph_manifest_summary,
    plot_timing_stats,
    timing_stats_dataframe,
    xglue_spec,
)

configure_notebook_style()
spec = xglue_spec()
GRAPH_TYPES = ["ast", "cfg", "ddg", "pdg", "cpg"]
ARTIFACT_DIR = notebook_output_dir(spec)
print(f"Project root: {PROJECT_ROOT}")
print(f"Data dir: {spec.data_dir}")
print(f"Output root: {spec.output_root}")
print(f"Notebook artifacts: {ARTIFACT_DIR}")


## Pipeline Validation


In [ ]:
display_pipeline_validation(spec, graph_types=GRAPH_TYPES, sample_label=1)


## Timing and Coverage


In [ ]:
timing_df = timing_stats_dataframe(spec)
if not timing_df.empty:
    display(timing_df)
    plot_timing_stats(spec)
    plot_graph_coverage_from_timing(spec)
else:
    print("No timing_stats.json found yet.")


## Similarity Distributions


In [ ]:
if spec.features_manifest.exists():
    score_df = display_similarity_distribution_report(
        spec,
        sample_size=1000,
        graph_types=GRAPH_TYPES,
        metrics=["pss"],
    )
    score_df.head()
else:
    print(f"Run 03_extract_spectral_features.py first. Missing: {spec.features_manifest}")


## PSS and Wasserstein Distribution Check


In [ ]:
if spec.features_manifest.exists():
    metric_df = display_statistical_distribution_plots(
        spec,
        sample_size=1000,
        graph_types=GRAPH_TYPES,
        seed=42,
    )
    metric_df.head()
else:
    print(f"Run 03_extract_spectral_features.py first. Missing: {spec.features_manifest}")
